In [ ]:
import numpy as np

def minimum_image(dr, box_length):
    return dr - box_length * np.round(dr / box_length)

def pair_energy_wca(r2, sigma=1.0, epsilon=1.0):
    rc = 2 ** (1/6) * sigma
    rc2 = rc * rc
    if r2 >= rc2:
        return 0.0
    inv_r2 = (sigma * sigma) / r2
    inv_r6 = inv_r2 ** 3
    inv_r12 = inv_r6 ** 2
    return 4 * epsilon * (inv_r12 - inv_r6) + epsilon

def total_energy(x, box_length, sigma=1.0, epsilon=1.0):
    n = x.shape[0]
    e = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            dr = minimum_image(x[i] - x[j], box_length)
            r2 = np.dot(dr, dr)
            e += pair_energy_wca(r2, sigma=sigma, epsilon=epsilon)
    return e

def random_initial_positions(n_particles, box_length, min_dist=0.8, seed=0):
    rng = np.random.default_rng(seed)
    pts = []
    while len(pts) < n_particles:
        cand = rng.uniform(0, box_length, size=2)
        ok = True
        for p in pts:
            dr = minimum_image(cand - p, box_length)
            if np.linalg.norm(dr) < min_dist:
                ok = False
                break
        if ok:
            pts.append(cand)
    return np.array(pts)

def metropolis_2d(
    n_solvent=32,
    box_length=6.0,
    n_steps=200000,
    save_every=100,
    step_size=0.15,
    beta=1.0,
    sigma=1.0,
    epsilon=1.0,
    seed=0,
):
    rng = np.random.default_rng(seed)
    n_particles = n_solvent + 1  # 1 solute
    x = random_initial_positions(n_particles, box_length, seed=seed)
    e = total_energy(x, box_length, sigma=sigma, epsilon=epsilon)

    frames = []
    accept = 0

    for t in range(n_steps):
        i = rng.integers(n_particles)
        old = x[i].copy()

        x[i] = (x[i] + rng.normal(scale=step_size, size=2)) % box_length
        e_new = total_energy(x, box_length, sigma=sigma, epsilon=epsilon)
        dE = e_new - e

        if dE <= 0 or rng.random() < np.exp(-beta * dE):
            e = e_new
            accept += 1
        else:
            x[i] = old

        if t % save_every == 0:
            frames.append(x.copy())

    return np.array(frames), accept / n_steps

In [4]:
frames, acc = metropolis_2d()
np.save("lj_2d_data.npy", frames)
print(frames.shape, acc)

(2000, 33, 2) 0.19886
